# PyTorch Experience · 02
## `model(x)` vs `model.forward(x)` · Forward Hooks

**One glance:** Both can run the same forward computation. **`model(x)` goes through PyTorch's `nn.Module` call mechanism**, including registered hooks on that module. **`model.forward(x)` calls the method directly**, bypassing those hooks.

**Official docs:** [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) · [register_forward_hook](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook) · [register_forward_pre_hook](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_pre_hook)

[← Pytorch Experience](https://lawson-dong.vercel.app/coding/pytorch-experience)

### 1 · See the difference

![Diagram comparing model(x) and model.forward(x)](https://raw.githubusercontent.com/Lawson-Dong/personal-website/main/public/images/pytorch-model-call-vs-forward.svg)

| Call | What happens? |
|---|---|
| `model(x)` | Runs `nn.Module` call handling, then `forward()` and registered hooks |
| `model.forward(x)` | Calls `forward()` directly; skips hooks registered **on `model` itself** |

**Rule: define `forward()`; normally call `model(x)`.**

### 2 · Two calls, same calculation

This simple model doubles a tensor. Run the cell—no GPU or downloads needed.

In [ ]:
import torch
from torch import nn

class Double(nn.Module):
    def forward(self, x):
        return x * 2

model = Double()
x = torch.tensor([1., 2.])

print('model(x):        ', model(x))
print('model.forward(x):', model.forward(x))

### 3 · Add a forward hook

A **forward hook** is a function PyTorch calls **after** a module's `forward()` has run. It receives `(module, inputs, output)`.

You can use it to **observe/save an output without editing `forward()`**. A hook *can* also replace an output if it explicitly returns a replacement, but the hook below only prints.

In [ ]:
def show_hook(module, inputs, output):
    print('HOOK ran! Output:', output)

handle = model.register_forward_hook(show_hook)

print('1) model(x)')
model(x)            # Hook runs

print('2) model.forward(x)')
model.forward(x)    # Hook on model does NOT run

### 4 · Remove the hook

Hooks stay registered until you remove them. Calling `handle.remove()` cleans up.

In [ ]:
handle.remove()
print('After removing the hook:')
model(x)  # No hook message

### Remember

- **`model(x)`** → preferred: it supports PyTorch's registered module hooks.
- **`model.forward(x)`** → direct method call; skips the *current module's* hook wrapper. Hooks on submodules called inside `forward()` may still run.
- **Forward hook** → runs after forward, useful for examining intermediate outputs. A read-only hook doesn't change the numeric result, but it can add memory/time overhead.
- **Neither call automatically turns on evaluation mode or disables gradient tracking.**

**References**

1. [PyTorch `torch.nn.Module` — calling a module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)
2. [PyTorch `register_forward_hook()` — signature and removable handle](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_hook)
3. [PyTorch `register_forward_pre_hook()` — before-forward hooks](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_forward_pre_hook)